# MiniMax-H3 Inference (DiffSynth-Studio, A100 80GB-aware)

Everyday image-to-video generation on an **NVIDIA A100 80GB** Colab runtime.
Supports four model profiles (Ticket set: `upgrade_tickets/`) ranging from a
fully GPU-resident fast NF4 path to a managed max-quality BF16 path -- see
the "Model profile" dropdown and the benchmark section near the end of this
notebook for measured tradeoffs. Verified against DiffSynth-Studio commit
`6343deda06b3e09efc9b1ce23c135c35a341d143` (pinned explicitly, not a blind
`git pull` -- see the "2. Environment setup" cells) -- see
`docs/diffsynth_h3_api_notes.md` in the repo for the full investigation
history.

This notebook is the everyday generation workflow (base H3, or H3 + a LoRA
you've already trained). For training a new LoRA, use
`H3_LoRA_Training.ipynb` instead -- that notebook still produces its own
per-checkpoint validation videos during training.

**Note on "sound prompt":** the current `MiniMaxH3Pipeline.__call__` has a
single `prompt` (plus optional `negative_prompt`) -- there is no separate
sound/dialogue prompt parameter. Any spoken dialogue should be written
directly into the main prompt (this matches how the official DiffSynth-Studio
examples do it, e.g. `...she is speaking in english: "..."`). This notebook
does not invent a field that doesn't exist upstream.

**Note on Turbo LoRAs:** Turbo Speed LoRAs only work correctly on the
`A100_FAST_NF4` and `COMPAT_NF4_MANAGED` profiles -- they are confirmed to
corrupt output on the BF16-DiT profiles (`A100_MAX_QUALITY`,
`A100_HYBRID_EXPERIMENTAL`). The Gradio UI blocks this combination outright;
see section 8 for the measured details.


## 1. Runtime check

In [ ]:
import subprocess, sys, shutil

print("=== nvidia-smi ===")
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30).stdout)
except Exception as e:
    print("nvidia-smi failed:", e)

import torch
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("device:", props.name, "| total VRAM GB:", round(props.total_memory / (1024**3), 2))
else:
    print("WARNING: no GPU. In Colab: Runtime > Change runtime type > GPU, then re-run this cell.")

total, used, free = shutil.disk_usage("/")
print("disk free GB:", round(free/(1024**3), 2))


## 2. Environment setup

Mounts Drive (for cached models and trained LoRA checkpoints -- both are
too large/valuable to re-fetch or lose every session), clones/installs
DiffSynth-Studio, and installs Gradio for the UI. Generated videos are
kept local to this Colab session instead (see the "Browse generated
videos" cell near the end) -- nothing you generate here is written to
Drive automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)

import os

DRIVE_ROOT = "/content/drive/MyDrive/minimax-h3"
CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
MODELS_CACHE_DIR = os.path.join(DRIVE_ROOT, "models")

# Generated videos stay on the session's local disk, not Drive -- they're
# scratch output to review/download per-session, not something this
# notebook persists for you. They're gone when the runtime resets or
# disconnects, so download anything you want to keep (see the "Browse
# generated videos" cell below, or the Colab file browser on the left).
GENERATIONS_DIR = "/content/generations"

for path in [CHECKPOINTS_DIR, MODELS_CACHE_DIR, GENERATIONS_DIR]:
    os.makedirs(path, exist_ok=True)
    print("ok:", path)

os.environ.setdefault("MODELSCOPE_CACHE", os.path.join(MODELS_CACHE_DIR, "modelscope_cache"))
os.environ.setdefault("HF_HOME", os.path.join(MODELS_CACHE_DIR, "huggingface_cache"))

# DiffSynth-Studio caches to a cwd-relative ./models/<model_id>/... path by
# default, not governed by MODELSCOPE_CACHE/HF_HOME above. Override to an
# absolute, Drive-backed path so this notebook and H3_LoRA_Training.ipynb
# share one model cache regardless of process/cwd, and it survives runtime
# restarts. Verified against commit b8e3811 (core/loader/config.py).
os.environ.setdefault("DIFFSYNTH_MODEL_BASE_PATH", os.path.join(MODELS_CACHE_DIR, "diffsynth_models"))


In [ ]:
REPO_DIR = "/content/DiffSynth-Studio"

# Pinned (not a blind `git pull`) to a commit verified 2026-08-14 to contain every upstream
# example this notebook's A100 profiles depend on: MiniMax-H3-Pruned-FL2VA.py,
# MiniMax-H3-Pruned-NF4-FL2VA.py, MiniMax-H3-Text-Embeddings.py, MiniMax-H3-FL2VA-Turbo.py,
# MiniMax-H3-Retake.py. To deliberately move forward: bump this SHA, re-run this cell, and
# re-verify those example files still exist and still match this notebook's assumptions
# before trusting new behavior.
DIFFSYNTH_COMMIT = "6343deda06b3e09efc9b1ce23c135c35a341d143"

def run(cmd, cwd=None, check=True):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        if check:
            raise RuntimeError(f"command failed: {' '.join(cmd)}")
    return result

if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "https://github.com/modelscope/DiffSynth-Studio.git", REPO_DIR])
run(["git", "fetch", "origin"], cwd=REPO_DIR)
run(["git", "checkout", "--detach", DIFFSYNTH_COMMIT], cwd=REPO_DIR)

commit_sha = run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR).stdout.strip()
print("DiffSynth-Studio commit (pinned):   ", DIFFSYNTH_COMMIT)
print("DiffSynth-Studio commit (checked out):", commit_sha)
if commit_sha != DIFFSYNTH_COMMIT:
    raise RuntimeError(
        f"Checked-out commit {commit_sha} does not match pinned DIFFSYNTH_COMMIT "
        f"{DIFFSYNTH_COMMIT}. Fix the pin or the checkout before continuing."
    )

run(["pip", "install", "-e", ".[all]", "--quiet"], cwd=REPO_DIR)

# The editable-install .pth hook DiffSynth-Studio just registered isn't
# picked up by this already-running kernel (site.py already ran at
# startup) -- "import diffsynth" fails with ModuleNotFoundError in later
# cells otherwise, even though the pip install above succeeded.
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
run(["pip", "install", "gradio", "--quiet"], check=True)


In [ ]:
import dataclasses
import torch

@dataclasses.dataclass
class RuntimeCapabilities:
    gpu_name: str
    total_vram_gib: float
    free_vram_gib: float
    compute_capability: tuple
    bf16_supported: bool
    torch_version: str
    cuda_version: str
    is_a100_80gb: bool
    diffsynth_commit: str
    model_cache_dir: str
    generations_dir: str
    vram_headroom_gib: float = 2.0

def detect_runtime_capabilities(diffsynth_commit, model_cache_dir, generations_dir, vram_headroom_gib=2.0):
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU available -- this notebook requires a GPU Colab runtime.")
    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    free_vram_bytes, _ = torch.cuda.mem_get_info("cuda")
    free_vram_gib = free_vram_bytes / (1024 ** 3)
    compute_capability = torch.cuda.get_device_capability(0)
    bf16_supported = torch.cuda.is_bf16_supported()
    # Key on VRAM size, not just the "A100" substring -- the 40GB variant must NOT pass this
    # gate, since the A100-80-specific resident/hybrid profiles assume ~80GB of headroom.
    is_a100_80gb = ("A100" in gpu_name) and (total_vram_gib >= 75) and bf16_supported
    return RuntimeCapabilities(
        gpu_name=gpu_name, total_vram_gib=total_vram_gib, free_vram_gib=free_vram_gib,
        compute_capability=compute_capability, bf16_supported=bf16_supported,
        torch_version=torch.__version__, cuda_version=torch.version.cuda,
        is_a100_80gb=is_a100_80gb, diffsynth_commit=diffsynth_commit,
        model_cache_dir=model_cache_dir, generations_dir=generations_dir,
        vram_headroom_gib=vram_headroom_gib,
    )

RUNTIME = detect_runtime_capabilities(commit_sha, MODELS_CACHE_DIR, GENERATIONS_DIR)
print(
    f"{RUNTIME.gpu_name} / {RUNTIME.total_vram_gib:.1f} GiB total / "
    f"{RUNTIME.free_vram_gib:.1f} GiB free / "
    f"BF16 {'supported' if RUNTIME.bf16_supported else 'NOT supported'} / "
    f"compute capability {RUNTIME.compute_capability} / "
    f"torch {RUNTIME.torch_version} / cuda {RUNTIME.cuda_version}"
)
print(
    "A100 80GB-class optimizations:",
    "ENABLED" if RUNTIME.is_a100_80gb else "DISABLED (falling back to managed/compatibility profiles)",
)


Optional: persist a fresh benchmark JSON to Drive (Ticket 006) using the
measured values recorded during this notebook's A100 80GB validation pass.
See section 8 near the end of this notebook for the human-readable summary
and the key Turbo/BF16-DiT incompatibility finding.

In [ ]:
import json, time

BENCHMARK_DIR = os.path.join(DRIVE_ROOT, "experiments", "a100-80gb-inference-benchmarks")
os.makedirs(BENCHMARK_DIR, exist_ok=True)

benchmark = {
    "timestamp": time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()),
    "diffsynth_commit": DIFFSYNTH_COMMIT,
    "runtime": dict(
        gpu_name=RUNTIME.gpu_name, total_vram_gib=RUNTIME.total_vram_gib,
        torch_version=RUNTIME.torch_version, cuda_version=RUNTIME.cuda_version,
        is_a100_80gb=RUNTIME.is_a100_80gb,
    ),
    "results": [
        {
            "profile": "A100_FAST_NF4",
            "resident": True,
            "resolution": "832x480", "num_frames": 39,
            "load_seconds_cold_drive": 847.8,
            "load_seconds_warm": 39.5,
            "vram_allocated_gib": 27.8,
            "base_50step_gen_seconds": 68.0,
            "base_50step_rate_s_per_it": 1.37,
            "turbo_8step_lora_gen_seconds": 14.6,
            "turbo_lora_compatible": True,
            "anime_lora_compatible": True,
            "notes": "Fastest profile. No warm-up penalty observed. Recommended default for interactive/preview use.",
        },
        {
            "profile": "COMPAT_NF4_MANAGED",
            "resident": False,
            "resolution": "832x480", "num_frames": 39,
            "load_seconds_warm": 14.0,
            "vram_allocated_gib": 78.2,
            "notes": "Same weights as Fast NF4 but CPU-staged. Only useful as an OOM fallback -- do not use as a normal default on an A100 80GB, since Fast NF4 fits fully resident.",
        },
        {
            "profile": "A100_MAX_QUALITY",
            "resident": False,
            "resolution": "1344x768", "num_frames": 39,
            "load_seconds_cold_drive": 908.4,
            "vram_allocated_gib": 0.0,
            "vram_note": "0 GiB allocated at load-complete -- MANAGED_VRAM_CONFIG keeps all ~105GB of BF16 weights on CPU/host RAM until actively used per-layer during the forward pass.",
            "base_50step_gen_seconds_total": 1215.4,
            "base_50step_denoise_seconds": 621.0,
            "base_50step_denoise_rate_s_per_it_avg_incl_warmup": 12.42,
            "base_50step_denoise_rate_s_per_it_steady_state": 4.25,
            "first_call_warmup_seconds_observed": "~420s (7 min) before the first denoising step showed any progress -- one-time cost of paging ~105GB of components from Drive-backed host RAM into use for the first time this session",
            "turbo_lora_compatible": False,
            "turbo_lora_failure_mode": "Not tested directly on Max Quality (tested on Hybrid, which shares the same pruned BF16 DiT) -- assumed incompatible for the same reason (see Hybrid results). Do not use Turbo LoRAs with any BF16-DiT profile.",
            "anime_lora_compatible": "assumed compatible (not explicitly re-tested on Max Quality; confirmed on Hybrid, which shares the identical DiT)",
            "notes": "Coherent, high-quality 1344x768 output confirmed. ~20 min per generation is impractically slow for interactive use -- reserve for final-quality renders only, not iteration.",
        },
        {
            "profile": "A100_HYBRID_EXPERIMENTAL",
            "resident": True,
            "resolution": "1344x768", "num_frames": 39,
            "load_seconds_cold_drive": 1087.7,
            "vram_allocated_gib": 53.6,
            "vram_reserved_gib": 53.9,
            "vram_free_after_load_gib": 24.9,
            "base_50step_gen_seconds": 218.08,
            "base_50step_denoise_seconds": 212.0,
            "base_50step_denoise_rate_s_per_it": 4.25,
            "turbo_lora_compatible": False,
            "turbo_lora_failure_mode": (
                "CONFIRMED LIVE: Turbo 8-step LoRA (strength=0.0625, steps=8, flow_shift=12.0 -- its "
                "correct verified recipe) structurally patches all 208/208 target tensors with no "
                "load-time error, but produces a corrupted output: a large solid-color circular "
                "watermark/glyph blob replacing the center of the frame. Reproduced identically with "
                "Turbo alone (no anime LoRA stacked). Root cause: Turbo's LoRA delta weights were "
                "learned specifically against the NF4-quantized pruned DiT's weights; applying the "
                "identical delta to the full-precision BF16 pruned DiT (same architecture, different "
                "precision) is invalid despite matching tensor shapes/names."
            ),
            "anime_lora_compatible": True,
            "anime_lora_note": "200/200 tensors patched, clean coherent output, visually consistent with Max Quality's output at the same seed/prompt.",
            "notes": (
                "5.6x faster than Max Quality (218s vs 1215s) with visually comparable quality at the "
                "same seed/prompt, and no first-call warm-up penalty observed (unlike Max Quality). "
                "Meets promotion criteria for base generation + ordinary style LoRAs: fits at 1344x768, "
                "24.9 GiB safety margin, coherent output. FAILS promotion criteria for Turbo Speed "
                "LoRAs specifically -- remains EXPERIMENTAL and gated behind explicit UI selection; "
                "Turbo LoRA use is blocked outright on this profile in the Gradio UI (Ticket 005)."
            ),
        },
    ],
    "kohya_lora_format_bug": {
        "summary": "Pre-existing bug (not caused by this A100 refactor) found and fixed during LoRA compatibility testing.",
        "detail": (
            "DiffSynth-Studio's MiniMaxH3LoRALoader only recognizes two LoRA key-naming conventions: "
            "its own native dotted format, and lightx2v/Diffusers-style separate-QKV naming "
            "('transformer_blocks.N.attn.to_q.lora_A.default.weight'). A third, common community "
            "format -- kohya/sd-scripts flat underscore-joined names "
            "('lora_unet_blocks_N_attn_out_proj.lora_down.weight', already-fused qkv_proj) -- is not "
            "recognized at all. Loading one silently reports '0 tensors are patched' with zero visual "
            "effect, no error. The community anime style LoRA (Inner-Reflections/MiniMax-H3-Looping-"
            "Sketch-Anime) uses exactly this format and was never actually applying in this notebook "
            "prior to this fix, on any checkpoint variant. Fixed via a small kohya-format detector + "
            "key converter (convert_kohya_to_diffsynth_format) wired into set_loras()."
        ),
    },
}

out_path = os.path.join(BENCHMARK_DIR, f"{benchmark['timestamp']}.json")
with open(out_path, "w") as f:
    json.dump(benchmark, f, indent=2)
print("saved benchmark:", out_path)


## 3. Model setup

Loads the MiniMax-H3-NF4 base checkpoint. Kept in a module-level variable so
switching LoRAs doesn't require reloading the ~14GB of base weights each
time.

In [ ]:
import gc
import time
import re
import torch
from diffsynth.pipelines.minimax_h3_audio_video import MiniMaxH3Pipeline, ModelConfig

# Two device policies, reused across every model profile below. MANAGED stages weights
# through CPU (safe on any GPU, including low-VRAM ones). RESIDENT keeps everything on CUDA
# -- only valid once Ticket 001's RUNTIME.is_a100_80gb gate has confirmed there's room.
MANAGED_VRAM_CONFIG = dict(
    offload_dtype=torch.bfloat16, offload_device="cpu",
    onload_dtype=torch.bfloat16, onload_device="cpu",
    preparing_dtype=torch.bfloat16, preparing_device="cuda",
    computation_dtype=torch.bfloat16, computation_device="cuda",
)
RESIDENT_CUDA_VRAM_CONFIG = dict(
    offload_dtype=torch.bfloat16, offload_device="cuda",
    onload_dtype=torch.bfloat16, onload_device="cuda",
    preparing_dtype=torch.bfloat16, preparing_device="cuda",
    computation_dtype=torch.bfloat16, computation_device="cuda",
)

# Explicit rather than relying on MiniMaxH3Pipeline.from_pretrained's implicit default
# (which happens to be this same value) -- Ticket 002 requirement.
PROCESSOR_CONFIG = ModelConfig(model_id="MiniMax/MiniMax-H3", origin_file_pattern="FL2VA/processor/")

def _cfg(model_id, pattern, vram_config):
    return ModelConfig(model_id=model_id, origin_file_pattern=pattern, **vram_config)

# Centralized model-profile registry. Every checkpoint ID / file pattern / device policy in
# this notebook lives here, not scattered across cells. Verified against the actual upstream
# example scripts at DIFFSYNTH_COMMIT (MiniMax-H3-Pruned-FL2VA.py, MiniMax-H3-Pruned-NF4-FL2VA.py,
# MiniMax-H3-Text-Embeddings.py) rather than guessed.
MODEL_PROFILES = {
    "A100_MAX_QUALITY": dict(
        label="A100 80GB -- Max Quality (pruned BF16 DiT + full BF16 encoder)",
        experimental=False,
        requires_a100_80gb=True,
        default_resolution=(1344, 768),
        vram_config=MANAGED_VRAM_CONFIG,  # BF16 encoder + pruned BF16 DiT exceed 80GB together resident
        vram_limit_policy="headroom",
        component_configs=lambda vc: [
            _cfg("MiniMax/MiniMax-H3", "FL2VA/text_encoder/model*.safetensors", vc),
            _cfg("Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_fl2va_pruned_bf16.safetensors", vc),
            _cfg("MiniMax/MiniMax-H3", "FL2VA/video_vae/source/model.safetensors", vc),
            _cfg("MiniMax/MiniMax-H3", "FL2VA/audio_vae/model.safetensors", vc),
        ],
    ),
    "A100_FAST_NF4": dict(
        label="A100 80GB -- Fast (pruned NF4, GPU-resident)",
        experimental=False,
        requires_a100_80gb=True,
        default_resolution=(832, 480),
        vram_config=RESIDENT_CUDA_VRAM_CONFIG,
        vram_limit_policy="none",  # vram_limit=None -- only reached after the A100-80 gate passes
        component_configs=lambda vc: [
            _cfg("DiffSynth-Studio/MiniMax-H3-NF4", "minimax-h3-fl2va-pruned-nf4.safetensors", vc),
            _cfg("DiffSynth-Studio/MiniMax-H3-NF4", "minimax-h3-text-encoder-nf4.safetensors", vc),
            _cfg("DiffSynth-Studio/MiniMax-H3-NF4", "video_vae_nf4.safetensors", vc),
            _cfg("DiffSynth-Studio/MiniMax-H3-NF4", "audio_vae_nf4.safetensors", vc),
        ],
    ),
    "COMPAT_NF4_MANAGED": dict(
        label="Compatibility -- pruned NF4, managed/offloaded (any GPU)",
        experimental=False,
        requires_a100_80gb=False,
        default_resolution=(832, 480),
        vram_config=MANAGED_VRAM_CONFIG,
        vram_limit_policy="headroom",
        component_configs=lambda vc: [
            _cfg("DiffSynth-Studio/MiniMax-H3-NF4", "minimax-h3-fl2va-pruned-nf4.safetensors", vc),
            _cfg("DiffSynth-Studio/MiniMax-H3-NF4", "minimax-h3-text-encoder-nf4.safetensors", vc),
            _cfg("DiffSynth-Studio/MiniMax-H3-NF4", "video_vae_nf4.safetensors", vc),
            _cfg("DiffSynth-Studio/MiniMax-H3-NF4", "audio_vae_nf4.safetensors", vc),
        ],
    ),
    # Ticket 004: pruned BF16 DiT (repeated-denoising precision) + NF4 encoder/VAEs
    # (one-pass conditioning, quantization-tolerant). Not the official reference combo --
    # experimental until benchmarked (Ticket 004/006) and gated behind an explicit UI toggle.
    "A100_HYBRID_EXPERIMENTAL": dict(
        label="A100 80GB -- Hybrid BF16 DiT + NF4 Encoder (experimental)",
        experimental=True,
        requires_a100_80gb=True,
        default_resolution=(1344, 768),
        vram_config=RESIDENT_CUDA_VRAM_CONFIG,
        vram_limit_policy="none",
        component_configs=lambda vc: [
            _cfg("Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_fl2va_pruned_bf16.safetensors", vc),
            _cfg("DiffSynth-Studio/MiniMax-H3-NF4", "minimax-h3-text-encoder-nf4.safetensors", vc),
            _cfg("DiffSynth-Studio/MiniMax-H3-NF4", "video_vae_nf4.safetensors", vc),
            _cfg("DiffSynth-Studio/MiniMax-H3-NF4", "audio_vae_nf4.safetensors", vc),
        ],
    ),
}

_loaded_pipe = None
_loaded_profile_name = None
_load_status = {}

def _release_pipeline():
    global _loaded_pipe, _loaded_profile_name
    if _loaded_pipe is not None:
        try:
            _loaded_pipe.clear_lora()
        except Exception:
            pass
        del _loaded_pipe
    _loaded_pipe = None
    _loaded_profile_name = None
    gc.collect()
    torch.cuda.empty_cache()

def get_pipeline(profile_name, runtime=None, allow_fallback=True):
    """Loads (or returns the already-cached) pipeline for a named MODEL_PROFILES entry.
    Only one pipeline is kept resident at a time -- switching profiles releases the old one
    (clear_lora + gc.collect + empty_cache) before building the new one, so 40GB-class
    pipelines don't accumulate in Python globals across profile switches."""
    global _loaded_pipe, _loaded_profile_name, _load_status
    runtime = runtime or RUNTIME
    if _loaded_pipe is not None and _loaded_profile_name == profile_name:
        return _loaded_pipe

    profile = MODEL_PROFILES[profile_name]
    if profile["requires_a100_80gb"] and not runtime.is_a100_80gb:
        raise ValueError(
            f"{profile_name} requires an A100 80GB-class runtime; current: "
            f"{runtime.gpu_name} ({runtime.total_vram_gib:.1f} GiB total)."
        )

    _release_pipeline()

    vram_config = profile["vram_config"]
    model_configs = profile["component_configs"](vram_config)
    vram_limit = None if profile["vram_limit_policy"] == "none" else (
        torch.cuda.mem_get_info("cuda")[1] / (1024 ** 3) - runtime.vram_headroom_gib
    )

    print(f"Loading {profile['label']}... this can take a while on first run (large download).")
    torch.cuda.reset_peak_memory_stats()
    load_start = time.time()
    try:
        pipe = MiniMaxH3Pipeline.from_pretrained(
            torch_dtype=torch.bfloat16, device="cuda",
            model_configs=model_configs,
            processor_config=PROCESSOR_CONFIG,
            vram_limit=vram_limit,
        )
    except torch.cuda.OutOfMemoryError as e:
        gc.collect()
        torch.cuda.empty_cache()
        if allow_fallback and profile_name != "COMPAT_NF4_MANAGED":
            print(f"WARNING: {profile_name} OOM'd on load ({e}). Falling back to COMPAT_NF4_MANAGED.")
            return get_pipeline("COMPAT_NF4_MANAGED", runtime=runtime, allow_fallback=False)
        raise
    load_seconds = time.time() - load_start
    torch.cuda.synchronize()

    _loaded_pipe = pipe
    _loaded_profile_name = profile_name
    _load_status = dict(
        profile_name=profile_name,
        resident="cuda" in (vram_config["offload_device"], vram_config["onload_device"]),
        load_seconds=load_seconds,
        vram_allocated_gib=torch.cuda.memory_allocated() / (1024 ** 3),
        vram_reserved_gib=torch.cuda.memory_reserved() / (1024 ** 3),
        free_vram_gib=torch.cuda.mem_get_info("cuda")[0] / (1024 ** 3),
    )
    print(
        f"Loaded {profile_name} in {load_seconds:.1f}s -- "
        f"allocated {_load_status['vram_allocated_gib']:.1f} GiB, "
        f"reserved {_load_status['vram_reserved_gib']:.1f} GiB, "
        f"free {_load_status['free_vram_gib']:.1f} GiB."
    )
    return pipe

# --- Kohya/sd-scripts-format LoRA support ---------------------------------------------
# DiffSynth-Studio's MiniMaxH3LoRALoader only auto-converts two key layouts: its own native
# dotted format, and lightx2v/Diffusers-style separate-QKV ("transformer_blocks.N.attn.to_q").
# A third, very common community format -- kohya/sd-scripts flat underscore-joined names
# ("lora_unet_blocks_0_attn_out_proj.lora_down.weight", already-fused qkv_proj) -- is not
# recognized at all. Loading one through pipe.load_lora() silently reports "0 tensors are
# patched" and has ZERO effect on generation -- no error, no warning that actually explains
# the real problem. Confirmed live: the community anime LoRA (Inner-Reflections'
# MiniMax-H3-Looping-Sketch-Anime) uses exactly this format and was never actually applying.
KOHYA_SUFFIX_MAP = {
    "attn_qkv_proj": "attn.qkv_proj",
    "attn_out_proj": "attn.out_proj",
    "mlp_fc1": "mlp.fc1",
    "mlp_fc2": "mlp.fc2",
}

def is_kohya_format(state_dict):
    return any(k.startswith("lora_unet_blocks_") for k in state_dict)

def convert_kohya_to_diffsynth_format(state_dict):
    """Rewrites kohya-style flat keys into DiffSynth's dotted module-path format
    (blocks.N.attn.qkv_proj.lora_down.weight, etc.) so the existing generic LoRA loader's
    dot-splitting logic -- which already works correctly for genuinely dotted key paths --
    can match them against the model's real module names."""
    converted = {}
    unmapped = []
    for key, tensor in state_dict.items():
        m = re.match(r"^lora_unet_blocks_(\d+)_(.+?)\.(lora_down|lora_up|alpha)(\..*)?$", key)
        if not m:
            unmapped.append(key)
            continue
        block_idx, suffix, kind, rest = m.groups()
        mapped_suffix = KOHYA_SUFFIX_MAP.get(suffix)
        if mapped_suffix is None:
            unmapped.append(key)
            continue
        converted[f"blocks.{block_idx}.{mapped_suffix}.{kind}{rest or ''}"] = tensor
    if unmapped:
        raise ValueError(
            f"convert_kohya_to_diffsynth_format: {len(unmapped)} unrecognized key(s), "
            f"e.g. {sorted(unmapped)[:5]} -- extend KOHYA_SUFFIX_MAP to cover them."
        )
    return converted

def load_lora_auto_format(pipe, module, lora_path, alpha):
    """Loads a LoRA checkpoint, transparently converting kohya-format state dicts before
    handing off to pipe.load_lora() -- DiffSynth-native and lightx2v-format LoRAs are
    unaffected and pass straight through."""
    from diffsynth.core.loader.file import load_state_dict
    state_dict = load_state_dict(lora_path, torch_dtype=pipe.torch_dtype, device=pipe.device)
    if is_kohya_format(state_dict):
        state_dict = convert_kohya_to_diffsynth_format(state_dict)
    pipe.load_lora(module, state_dict=state_dict, alpha=alpha)

_current_lora_paths = []

def set_loras(pipe, lora_specs):
    """lora_specs: iterable of (lora_path, strength) pairs. A lora_path of
    None/"" is skipped. LoRAs stack additively -- DiffSynth-Studio's
    load_lora() hotloads by appending to a list of LoRA A/B weight pairs
    rather than replacing (verified via inspect.getsource against the live
    pipeline), so a style LoRA (e.g. the anime LoRA) and a speed LoRA (e.g.
    the Turbo LoRA) can both be active at once. Always clears first so
    re-generating with different dropdown choices doesn't accumulate stale
    LoRAs left over from a previous run."""
    global _current_lora_paths
    pipe.clear_lora()
    _current_lora_paths = []
    for lora_path, strength in lora_specs:
        if lora_path:
            load_lora_auto_format(pipe, pipe.dit, lora_path, alpha=strength)
            _current_lora_paths.append(lora_path)

def list_lora_checkpoints():
    """Scan Drive for trained/downloaded LoRA checkpoints, most recent first."""
    found = []
    if not os.path.isdir(CHECKPOINTS_DIR):
        return found
    for experiment_name in sorted(os.listdir(CHECKPOINTS_DIR)):
        exp_dir = os.path.join(CHECKPOINTS_DIR, experiment_name)
        if not os.path.isdir(exp_dir):
            continue
        for fname in sorted(os.listdir(exp_dir)):
            if fname.endswith(".safetensors"):
                found.append(os.path.join(exp_dir, fname))
    return found


## 3b. Community LoRA: anime looping-sketch style

Downloads a ready-to-use community style LoRA from Hugging Face --
[`Inner-Reflections/MiniMax-H3-Looping-Sketch-Anime`](https://huggingface.co/Inner-Reflections/MiniMax-H3-Looping-Sketch-Anime),
a hand-drawn 2D anime look (rough outlines, flat colors, white outline).
No training required. Cached to Drive (`CHECKPOINTS_DIR`) so it only
downloads once, and shows up automatically in the "LoRA checkpoint"
dropdown below as `community-anime-looping-sketch`.

Suggested use: LoRA strength `0.75`-`1.25`, and include this in your
prompt: *"A hand-drawn 2D anime art style characterized by rough,
textured outlines, coloring technique is minimalist, utilizing flat
colors and white outline."* (Drop the word "anime" from that line for a
western-sketch look from the same LoRA.)


In [ ]:
from huggingface_hub import hf_hub_download
import shutil

ANIME_LORA_REPO = "Inner-Reflections/MiniMax-H3-Looping-Sketch-Anime"
ANIME_LORA_FILE = "minimax_h3_looping_sketch_anime_v1.safetensors"

community_lora_dir = os.path.join(CHECKPOINTS_DIR, "community-anime-looping-sketch")
os.makedirs(community_lora_dir, exist_ok=True)
anime_lora_path = os.path.join(community_lora_dir, ANIME_LORA_FILE)

if os.path.exists(anime_lora_path):
    print("already cached:", anime_lora_path)
else:
    print(f"downloading {ANIME_LORA_REPO} ({ANIME_LORA_FILE})...")
    cached_path = hf_hub_download(repo_id=ANIME_LORA_REPO, filename=ANIME_LORA_FILE)
    shutil.copy(cached_path, anime_lora_path)
    print("saved to:", anime_lora_path)


## 3c. Community LoRA: Turbo (fast) style

Downloads both verified checkpoints from
[`lightx2v/Minimax-h3-Turbo`](https://huggingface.co/lightx2v/Minimax-h3-Turbo) --
distilled LoRAs that trade some quality for speed (4 or 8 inference steps
instead of 50). No training required; cached to Drive so each only
downloads once, and both show up in the "Speed LoRA" dropdown below.

**Required settings per checkpoint** (getting these wrong produces
garbled/static output, not just lower quality):

| Checkpoint | Dropdown name | Inference steps | Flow shift | LoRA strength |
|---|---|---|---|---|
| Turbo 4-step v1.0 (768p) | `community-turbo-768p-4step` | 4 | 6.0 | 1.0 |
| Turbo 8-step v1.0 | `community-turbo-8step` | 8 | 12.0 (pipeline default) | **0.0625** |

**Why the strength differs between them:** DiffSynth-Studio's
`pipe.load_lora(module, lora_path, alpha=X)` treats `alpha` as a raw
multiplier on the LoRA weights, not the PEFT convention of
`alpha / rank`. Both checkpoints are LoRA rank 128. The 4-step
checkpoint's official reference script overrides `lora-alpha=128`
(`128/128 = 1.0` in PEFT terms), matching DiffSynth's raw multiplier --
so `strength=1.0` is correct there. The 8-step checkpoint's reference
script does *not* override `lora-alpha` (default `8`), so the
PEFT-equivalent scale is `8/128 = 0.0625`. Using `strength=1.0` on the
8-step checkpoint is a 16x overscale and produces static-noise output
that looks like a broken model, not a subtle quality issue.

**Turbo LoRAs only work on `A100_FAST_NF4` / `COMPAT_NF4_MANAGED`.**
Confirmed live: applying the exact verified recipe above to the
BF16-DiT profiles (`A100_MAX_QUALITY`, `A100_HYBRID_EXPERIMENTAL`)
structurally patches every target tensor with no load-time error, but
produces a corrupted watermark/glyph artifact instead of the requested
scene -- the Turbo LoRA's delta weights were learned specifically
against the NF4-quantized DiT and don't transfer to full BF16 precision
despite matching tensor shapes. The Gradio UI blocks this combination
outright (`TURBO_INCOMPATIBLE_PROFILES`). See section 8 (benchmark
results) and `docs/diffsynth_h3_api_notes.md` for the full investigation.


In [ ]:
from huggingface_hub import hf_hub_download
import shutil

def _download_turbo_lora(repo_id, filename, subdir):
    lora_dir = os.path.join(CHECKPOINTS_DIR, subdir)
    os.makedirs(lora_dir, exist_ok=True)
    lora_path = os.path.join(lora_dir, filename)
    if os.path.exists(lora_path):
        print("already cached:", lora_path)
    else:
        print(f"downloading {repo_id} ({filename})...")
        cached_path = hf_hub_download(repo_id=repo_id, filename=filename)
        shutil.copy(cached_path, lora_path)
        print("saved to:", lora_path)
    return lora_path

TURBO_LORA_REPO = "lightx2v/Minimax-h3-Turbo"

turbo_768p_4step_path = _download_turbo_lora(
    TURBO_LORA_REPO, "minimax_h3_fl2v_turbo_4step_v1.0_768p_bf16.safetensors", "community-turbo-768p-4step",
)
turbo_8step_path = _download_turbo_lora(
    TURBO_LORA_REPO, "minimax_h3_fl2v_turbo_8step_v1.0_bf16.safetensors", "community-turbo-8step",
)


## 4. Image-to-video input, prompt, and settings

The Gradio form below is the everyday interface. It supports:

- base H3 inference (no LoRA), and
- LoRA inference (select a trained checkpoint you produced with
  `H3_LoRA_Training.ipynb`).

`num_frames` is restricted to valid H3 values (`17n + 5`).

In [ ]:
import gradio as gr
from PIL import Image
from diffsynth.utils.data.audio_video import write_video_audio
import time, re

# 17n+5, n=2..20 -- up to 345 frames (~14.4s @ 24fps), the largest valid value at or under
# MiniMax-H3's native ~15s single-call ceiling. DiffSynth's own official retake example uses
# num_frames=175, well past the old 124 default this notebook used to be capped at.
H3_VALID_FRAME_COUNTS = [39, 56, 73, 90, 107, 124, 141, 158, 175, 192, 209, 226, 243, 260, 277, 294, 311, 328, 345]

RESOLUTION_PRESETS = {
    "Preview landscape -- 832x480": (832, 480),
    "Preview portrait -- 480x832": (480, 832),
    "Final landscape -- 1344x768": (1344, 768),
    "Final portrait -- 768x1344": (768, 1344),
    "Custom": None,
}

QUALITY_PRESETS = {
    "Draft -- Fast NF4, preview res": dict(profile="A100_FAST_NF4", resolution="Preview landscape -- 832x480"),
    "Turbo preview -- Fast NF4 + Turbo LoRA": dict(profile="A100_FAST_NF4", resolution="Preview landscape -- 832x480"),
    "Final -- Max Quality, 768p": dict(profile="A100_MAX_QUALITY", resolution="Final landscape -- 1344x768"),
    "Custom": None,
}

# Verified live (see docs/diffsynth_h3_api_notes.md): each Turbo checkpoint requires its own
# exact steps/shift/strength combo. Getting these wrong doesn't just lower quality -- it
# produces garbled/static output, so these are auto-applied by default rather than left to
# the user to remember.
TURBO_RECIPES = {
    "community-turbo-768p-4step": dict(steps=4, flow_shift=6.0, strength=1.0, label="Turbo 4-step v1.0 (768p)"),
    "community-turbo-8step": dict(steps=8, flow_shift=12.0, strength=0.0625, label="Turbo 8-step v1.0"),
}

# Confirmed live: Turbo Speed LoRAs are distilled against the NF4-quantized pruned DiT
# specifically. Applying the identical strength/steps/shift to the full-precision BF16 pruned
# DiT (Max Quality's and Hybrid's DiT) produces a corrupted watermark-like artifact, not a
# subtle quality loss -- despite the LoRA structurally patching all its target tensors
# (208/208) without any load-time error. Block it outright rather than let it silently corrupt.
TURBO_INCOMPATIBLE_PROFILES = {"A100_MAX_QUALITY", "A100_HYBRID_EXPERIMENTAL"}

def sanitize_for_filename(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", name).strip("-") or "lora"

def find_turbo_recipe(lora_path):
    if not lora_path:
        return None
    for key, recipe in TURBO_RECIPES.items():
        if key in lora_path:
            return recipe
    return None

def profile_choices_for_dropdown():
    return [(profile["label"], name) for name, profile in MODEL_PROFILES.items()]

def build_status_markdown(profile_name, height, width, steps, flow_shift, active_loras_desc, load_status, gen_seconds):
    profile = MODEL_PROFILES[profile_name]
    resident = load_status.get("resident") if load_status else None
    mem_mode = "GPU-resident" if resident else ("managed/CPU-staged" if resident is not None else "unknown")
    vram_line = ""
    if load_status:
        vram_line = (
            f"- VRAM: {load_status['vram_allocated_gib']:.1f} GiB allocated, "
            f"{load_status['free_vram_gib']:.1f} GiB free after load\n"
        )
    return (
        f"**Profile:** {profile['label']}{' *(experimental)*' if profile['experimental'] else ''}\n\n"
        f"- Memory mode: {mem_mode}\n"
        f"{vram_line}"
        f"- Resolution: {int(width)}x{int(height)} ({int(width)//32}x{int(height)//32} in 32px units)\n"
        f"- Steps: {steps} / Flow shift: {flow_shift}\n"
        f"- LoRAs: {active_loras_desc}\n"
        f"- Generation time: {gen_seconds:.1f}s\n"
    )

def apply_quality_preset(preset_name):
    cfg = QUALITY_PRESETS.get(preset_name)
    if cfg is None:
        return gr.update(), gr.update()
    return gr.update(value=cfg["profile"]), gr.update(value=cfg["resolution"])

def apply_resolution_preset(preset_name):
    dims = RESOLUTION_PRESETS.get(preset_name)
    if dims is None:
        return gr.update(), gr.update()
    width, height = dims
    return gr.update(value=height), gr.update(value=width)

def apply_turbo_recipe(speed_lora_choice, allow_manual_override):
    if allow_manual_override:
        return gr.update(), gr.update(), gr.update()
    recipe = find_turbo_recipe(speed_lora_choice)
    if recipe is None:
        return gr.update(), gr.update(), gr.update()
    return gr.update(value=recipe["steps"]), gr.update(value=recipe["flow_shift"]), gr.update(value=recipe["strength"])

def generate(
    profile_name, height, width, num_frames,
    style_lora_choice, style_lora_strength,
    speed_lora_choice, speed_lora_strength,
    num_inference_steps, flow_shift, seed, allow_manual_turbo_override,
    first_frame_img, prompt, negative_prompt,
    progress=gr.Progress(),
):
    if first_frame_img is None:
        raise gr.Error("Upload a first-frame image before generating.")
    if not prompt or not prompt.strip():
        raise gr.Error("Prompt is required.")
    height, width, num_frames = int(height), int(width), int(num_frames)
    if height % 32 != 0 or width % 32 != 0:
        raise gr.Error(f"Height and width must be multiples of 32 (got {width}x{height}).")
    if num_frames not in H3_VALID_FRAME_COUNTS:
        raise gr.Error(f"Frame count must be a valid H3 value (17n+5): {H3_VALID_FRAME_COUNTS}")

    speed_lora_path = None if speed_lora_choice in (None, "None", "") else speed_lora_choice
    recipe = find_turbo_recipe(speed_lora_path)

    if speed_lora_path and profile_name in TURBO_INCOMPATIBLE_PROFILES:
        raise gr.Error(
            f"The selected Speed LoRA is a Turbo checkpoint distilled against the NF4-quantized "
            f"DiT -- it's confirmed to produce corrupted output on {MODEL_PROFILES[profile_name]['label']}. "
            f"Use A100_FAST_NF4 or the compatibility profile instead, or remove the Speed LoRA."
        )

    effective_steps, effective_shift, effective_strength = num_inference_steps, flow_shift, speed_lora_strength
    if recipe and not allow_manual_turbo_override:
        effective_steps, effective_shift, effective_strength = recipe["steps"], recipe["flow_shift"], recipe["strength"]
    elif recipe and allow_manual_turbo_override:
        pass  # user explicitly wants manual control -- don't second-guess their values

    progress(0, desc=f"Loading {MODEL_PROFILES[profile_name]['label']} (first run can take a while)...")
    pipe = get_pipeline(profile_name)

    lora_specs = [
        (None if style_lora_choice in (None, "None", "") else style_lora_choice, style_lora_strength),
        (speed_lora_path, effective_strength),
    ]
    active_loras = [p for p, _ in lora_specs if p]
    progress(0, desc=f"Applying {len(active_loras)} LoRA(s)..." if active_loras else "Using base model (no LoRA)...")
    set_loras(pipe, lora_specs)

    first_frame = first_frame_img.convert("RGB")
    t0 = time.time()
    video, audio = pipe(
        prompt=prompt, negative_prompt=negative_prompt or " ",
        height=height, width=width, num_frames=num_frames,
        num_inference_steps=int(effective_steps), flow_shift=float(effective_shift), seed=int(seed),
        keyframes=[first_frame], keyframe_indices=[0],
        progress_bar_cmd=lambda it: progress.tqdm(it, desc="Generating (denoising)"),
    )
    gen_seconds = time.time() - t0

    progress(1, desc="Saving video...")
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    lora_tag = "no-lora" if not active_loras else "+".join(
        sanitize_for_filename(os.path.splitext(os.path.basename(p))[0]) for p in active_loras
    )
    out_name = f"{timestamp}_{profile_name}_{width}x{height}_{lora_tag}_seed{seed}.mp4"
    out_path = os.path.join(GENERATIONS_DIR, out_name)
    write_video_audio(video=video, audio=audio, output_path=out_path, fps=24, audio_sample_rate=pipe.audio_vae.sample_rate)

    active_loras_desc = ", ".join(
        f"{os.path.basename(p)}@{s}" for p, s in [(style_lora_choice, style_lora_strength), (speed_lora_path, effective_strength)] if p
    ) or "none"
    status_md = build_status_markdown(profile_name, height, width, effective_steps, effective_shift, active_loras_desc, _load_status, gen_seconds)
    print(f"saved {out_path} -- {status_md}")
    return out_path, status_md

with gr.Blocks(title="MiniMax-H3 Inference") as demo:
    gr.Markdown(
        "## MiniMax-H3 Inference (A100 80GB)\n"
        "Base H3 generation, or generation with up to two stacked LoRAs -- a **Style** LoRA "
        "(e.g. the anime looping-sketch LoRA) and a **Speed** LoRA (e.g. a Turbo LoRA). "
        "Selecting a recognized Turbo checkpoint auto-applies its verified steps/shift/strength "
        "recipe unless \"Advanced: allow manual Turbo override\" is checked. **Turbo LoRAs are "
        "not compatible with the Max Quality or Hybrid profiles** (confirmed to produce corrupted "
        "output) -- use Fast NF4 or Compatibility for Turbo generations."
    )
    with gr.Row():
        with gr.Column():
            quality_preset = gr.Dropdown(list(QUALITY_PRESETS.keys()), value="Custom", label="Quality preset")
            profile_name = gr.Dropdown(profile_choices_for_dropdown(), value="A100_FAST_NF4", label="Model profile")
            resolution_preset = gr.Dropdown(list(RESOLUTION_PRESETS.keys()), value="Preview landscape -- 832x480", label="Resolution preset")
            with gr.Accordion("Advanced / Custom resolution", open=False):
                height = gr.Number(value=480, precision=0, label="Height (must be a multiple of 32)")
                width = gr.Number(value=832, precision=0, label="Width (must be a multiple of 32)")
            num_frames = gr.Dropdown(H3_VALID_FRAME_COUNTS, value=124, label="Frame count (17n+5, up to 345 = ~14.4s @ 24fps)")
            with gr.Row():
                style_lora_choice = gr.Dropdown(
                    choices=["None"] + list_lora_checkpoints(), value="None",
                    label="Style LoRA (optional)",
                )
                style_lora_strength = gr.Slider(0.0, 2.0, value=1.0, step=0.05, label="Style LoRA strength")
            with gr.Row():
                speed_lora_choice = gr.Dropdown(
                    choices=["None"] + list_lora_checkpoints(), value="None",
                    label="Speed LoRA (optional, e.g. Turbo)",
                )
                speed_lora_strength = gr.Number(value=1.0, label="Speed LoRA strength (exact value matters)")
            allow_manual_turbo_override = gr.Checkbox(value=False, label="Advanced: allow manual Turbo override (skip auto-recipe)")
            refresh_btn = gr.Button("Refresh LoRA lists")
            first_frame_img = gr.Image(type="pil", label="First-frame image")
            prompt = gr.Textbox(label="Prompt", lines=4, placeholder="Describe the motion (and any spoken dialogue)...")
            negative_prompt = gr.Textbox(label="Negative prompt (optional)", value=" ")
            with gr.Row():
                num_inference_steps = gr.Number(value=50, precision=0, label="Inference steps")
                flow_shift = gr.Number(value=12.0, label="Flow shift")
                seed = gr.Number(value=42, precision=0, label="Seed")
            generate_btn = gr.Button("Generate", variant="primary")
        with gr.Column():
            output_video = gr.Video(label="Generated video")
            status_card = gr.Markdown(label="Run status")

    quality_preset.change(apply_quality_preset, inputs=[quality_preset], outputs=[profile_name, resolution_preset])
    resolution_preset.change(apply_resolution_preset, inputs=[resolution_preset], outputs=[height, width])
    speed_lora_choice.change(
        apply_turbo_recipe, inputs=[speed_lora_choice, allow_manual_turbo_override],
        outputs=[num_inference_steps, flow_shift, speed_lora_strength],
    )
    refresh_btn.click(
        lambda: (
            gr.update(choices=["None"] + list_lora_checkpoints()),
            gr.update(choices=["None"] + list_lora_checkpoints()),
        ),
        outputs=[style_lora_choice, speed_lora_choice],
    )
    generate_btn.click(
        generate,
        inputs=[profile_name, height, width, num_frames,
                style_lora_choice, style_lora_strength, speed_lora_choice, speed_lora_strength,
                num_inference_steps, flow_shift, seed, allow_manual_turbo_override,
                first_frame_img, prompt, negative_prompt],
        outputs=[output_video, status_card],
    )


## 5. Launch

`share=False` by default (no public link). Set `share=True` only if you
intentionally want a temporary public URL.

In [ ]:
import gradio.tunneling as tunneling

# The frpc share-tunnel binary caches under HF_HOME, which we point at Drive
# for the ~32GB model weights -- but Drive's FUSE mount can't mark files
# executable, so frpc fails with PermissionError if it lands there. Redirect
# just this small binary to local (ephemeral, re-fetched each session) disk.
tunneling.BINARY_PATH = os.path.join("/content/.frpc_cache", os.path.basename(tunneling.BINARY_PATH))
os.makedirs(os.path.dirname(tunneling.BINARY_PATH), exist_ok=True)

demo.launch(share=True, debug=False)


## 6. Output

Generated videos are written to `/content/generations/` on this Colab
session's local disk (not Drive), named
`<timestamp>_<checkpoint>_<lora>_seed<seed>.mp4`, and previewable directly
in the Gradio UI above. They do not persist past this session -- use the
next cell to browse/preview everything generated so far, or download files
you want to keep via the Colab file browser (folder icon, left sidebar)
before the runtime disconnects.

## 7. Browse generated videos

Run this any time (no need to relaunch Gradio) to list and preview every
video generated so far this session, most recent first. Videos are embedded
inline (base64), since Colab can't serve arbitrary local file paths directly.

In [ ]:
import glob
from IPython.display import display, Video

files = sorted(glob.glob(os.path.join(GENERATIONS_DIR, "*.mp4")), key=os.path.getmtime, reverse=True)
if not files:
    print("No generated videos yet -- run a generation in the Gradio UI above first.")
for f in files:
    size_mb = os.path.getsize(f) / (1024 ** 2)
    print(f"{os.path.basename(f)}  ({size_mb:.1f} MB)")
    display(Video(f, embed=True, width=480))


## 8. A100 80GB benchmark results

Measured live on an `NVIDIA A100-SXM4-80GB` runtime (Ticket 006). Run the cell
below to persist a fresh benchmark JSON to
`MyDrive/minimax-h3/experiments/a100-80gb-inference-benchmarks/`, or read
below for the summary from the run that shipped this notebook version.

| Profile | Resident | Load (cold) | Gen (39f, 50 steps) | Turbo LoRA | Style LoRA |
|---|---|---|---|---|---|
| `A100_FAST_NF4` | Yes | ~848s | ~68s (1.37s/it) | ✅ compatible | ✅ compatible |
| `COMPAT_NF4_MANAGED` | No | ~14s (warm) | fallback only | -- | -- |
| `A100_MAX_QUALITY` | No | ~908s | ~1215s (~20 min, incl. ~7 min one-time warm-up) | âŒ corrupts output | ✅ compatible |
| `A100_HYBRID_EXPERIMENTAL` | Yes | ~1088s | ~218s (4.25s/it, no warm-up) | âŒ corrupts output | ✅ compatible |

**Key finding:** Turbo Speed LoRAs are distilled specifically against the
NF4-quantized pruned DiT. Applying the identical (correct, verified) recipe
to the full-precision BF16 pruned DiT -- used by both Max Quality and Hybrid
-- structurally patches every target tensor with no load-time error, but
produces a corrupted watermark/glyph artifact instead of the requested scene.
This is blocked outright in the Gradio UI (`TURBO_INCOMPATIBLE_PROFILES`) --
use `A100_FAST_NF4` or `COMPAT_NF4_MANAGED` for Turbo generations.

**Recommendation:** `A100_FAST_NF4` for interactive/preview work (fastest,
Turbo-compatible). `A100_HYBRID_EXPERIMENTAL` for near-BF16 quality without
Max Quality's ~20-minute generation time, when Turbo isn't needed --
still labeled experimental pending broader validation. `A100_MAX_QUALITY`
only for final, non-interactive renders.
